# TPI — Detección de Parkinson mediante análisis tiempo-frecuencia de voz
**Señales y Sistemas 2026**


Cuando llegue PC-GITA: montar Drive y cambiar `DATA_DIR`.

---
 1. Instalación
 2. Imports y config
 3. Preprocesamiento
 4. Wavelet
 5. Features
 6. Dataset
 7. Pruebas

In [ ]:
!pip install librosa PyWavelets scipy matplotlib seaborn pandas -q  # instala todas las dependencias; -q suprime el log de instalacion

In [ ]:
import numpy as np                           # arrays N-dimensionales y algebra lineal/estadistica vectorizada
import pywt                                   # PyWavelets: banco de filtros DWT y su inversa (wavedec/waverec)
import librosa                                # carga audio .wav → array numpy y calcula STFT
import librosa.display                        # specshow: dibuja espectrogramas con ejes tiempo [s] / frecuencia [Hz]
import matplotlib.pyplot as plt               # crea figuras, subplots y guarda PNG
import seaborn as sns                         # graficos estadisticos avanzados (disponible para analisis exploratorio)
import pandas as pd                           # DataFrame tabular para acumular features y exportar CSV
from pathlib import Path                      # rutas multiplataforma como objetos; operador / concatena carpetas
from scipy.signal import find_peaks, hilbert  # find_peaks: maximos locales con restricciones; hilbert: señal analitica

print('Importaciones OK')  # confirma que todas las librerias estan instaladas y cargaron sin error

In [12]:
SR          = 44100   # Frecuencia de muestreo de PC-GITA [Hz]
NIVELES_DWT = 9       # Niveles de descomposición diádica → 10 arrays (cD1–cD9 + cA9)

# Bandas frecuenciales resultantes con SR = 44100 Hz y NIVELES_DWT = 9
#
#  Índice en coeffs │ Array  │ Banda [Hz]
# ─────────────────┼────────┼──────────────────────────────
#  coeffs[0]       │ cA9    │     0 –    43 Hz
#  coeffs[1]       │ cD9    │    43 –    86 Hz  ← jitter (D9)
#  coeffs[2]       │ cD8    │    86 –   172 Hz  ← jitter (D8)
#  coeffs[3]       │ cD7    │   172 –   344 Hz  ← jitter (D7)
#  coeffs[4]       │ cD6    │   344 –   689 Hz
#  coeffs[5]       │ cD5    │   689 –  1378 Hz
#  coeffs[6]       │ cD4    │  1378 –  2756 Hz  ← shimmer / energía (D4)
#  coeffs[7]       │ cD3    │  2756 –  5512 Hz  ← shimmer / energía (D3)
#  coeffs[8]       │ cD2    │  5512 – 11025 Hz  ← shimmer / energía (D2)
#  coeffs[9]       │ cD1    │ 11025 – 22050 Hz  ← shimmer / energía (D1)
#
# Índice de Dj en la lista:  idx = NIVELES_DWT + 1 - j  (ej: D1 → 9, D9 → 1)
# Jitter    → D7, D8, D9  (43–344 Hz, rango F0 de la voz humana)
# Shimmer   → D1–D4       (1378–22050 Hz, zona de ruido glótico)
# Energía   → D1–D4       (misma zona, relación E_alta / E_total)

print(f'SR          : {SR} Hz')
print(f'Niveles DWT : {NIVELES_DWT}  → {NIVELES_DWT + 1} arrays')

SR          : 44100 Hz
Niveles DWT : 9  → 10 arrays


## Modulo 1 — Carga y preprocesamiento

**Por que librosa:** PyWavelets solo trabaja con arrays numpy. Librosa hace el puente: lee el `.wav` del disco y lo entrega como array. Tambien recorta silencios con una sola llamada.

- Entrada: archivo `.wav`
- Salida: `np.ndarray` shape `(N,)`, float64, normalizado en [-1, 1]

In [13]:
def cargar_señal(path: Path, sr_objetivo: int = SR) -> tuple:
    """Carga .wav con librosa y normaliza en amplitud. PC-GITA ya esta a 44100 Hz."""
    y, sr = librosa.load(path, sr=sr_objetivo, mono=True)
    # librosa.load: lee el archivo .wav del disco
    # sr=sr_objetivo: remuestrea a 44100 Hz si el archivo tiene otra tasa
    # mono=True: promedia canales si el audio es estereo, devuelve un solo canal

    y = y / (np.max(np.abs(y)) + 1e-9)
    # normaliza la amplitud al rango [-1, 1] dividiendo por el maximo valor absoluto
    # el epsilon 1e-9 evita division por cero en señales completamente silenciosas

    return y.astype(np.float64), sr
    # convierte a float64 para mayor precision numerica en los calculos de la DWT


def recortar_silencios(y: np.ndarray, sr: int, top_db: int = 20) -> np.ndarray:
    """Elimina tramos con energia < top_db dB bajo el maximo.

    CORRECCION aplicada: top_db reducido de 30 a 20 dB.
    Con 30 dB se eliminaban tramos de voz debil que en voces patologicas
    son diagnosticamente relevantes: la hipofonia (voz apagada de baja intensidad)
    es un sintoma tipico de la disartria hipocinetica del Parkinson.
    Con 20 dB solo se eliminan silencios reales, conservando la voz debil.
    """
    intervalos = librosa.effects.split(y, top_db=top_db)
    # librosa.effects.split: detecta todos los intervalos donde la señal supera el umbral de energia
    # top_db=20: conserva cualquier tramo cuya energia este dentro de los 20 dB del pico maximo
    # retorna una lista de pares [inicio, fin] expresados en numero de muestra

    if len(intervalos) == 0:
        return y
    # caso borde: si no hay ningun intervalo activo (señal totalmente silenciosa), devuelve la señal original intacta

    return np.concatenate([y[i:f] for i, f in intervalos])
    # extrae cada tramo activo con slicing y los concatena en una sola señal continua sin silencios


def segmentar(y: np.ndarray, sr: int,
              dur_seg: float = 2.0, solapamiento: float = 0.5) -> list:
    """Ventanas de 2 s con 50% de solapamiento -> 88200 muestras por ventana.

    ADVERTENCIA — riesgo de fuga de datos (data leakage):
    Los segmentos solapados de un mismo archivo comparten muestras entre si.
    Si el split train/test se hace a nivel de SEGMENTO, muestras casi identicas
    aparecen simultaneamente en train y en test, inflando artificialmente las metricas.
    SIEMPRE hacer el split agrupando por identificador de PACIENTE/ARCHIVO,
    y solo segmentar DESPUES de separar los conjuntos.
    """
    largo = int(dur_seg * sr)
    # largo: cantidad de muestras por ventana = 2.0 s * 44100 Hz = 88200 muestras

    paso = int(largo * (1.0 - solapamiento))
    # paso: cuantas muestras se avanza entre ventanas consecutivas
    # con solapamiento=0.5 el paso es 44100 muestras (1 segundo), cada ventana comparte la mitad con la anterior

    return [y[i : i + largo] for i in range(0, len(y) - largo + 1, paso)]
    # genera la lista de segmentos deslizando la ventana de largo a largo
    # la condicion len(y) - largo + 1 garantiza que el ultimo segmento este completo


print('Modulo 1 OK — cargar_señal | recortar_silencios | segmentar')


Modulo 1 OK — cargar_señal | recortar_silencios | segmentar


## Modulo 2 — Descomposicion wavelet

**DWT:** red diadica (s = 2^-j, t = k·2^-j). Solo las aproximaciones se descomponen en cada iteracion.

Wavelet fija: **Daubechies-4 (db4)**. 9 niveles → 10 arrays (cA9 + cD1–cD9).

In [ ]:
def dwt_multirresolucion(señal: np.ndarray) -> list:
    """
    DWT multinivel db4 por red diádica (s = 2^-j, τ = k·2^-j).
    Wavelet fija: Daubechies-4. Niveles: NIVELES_DWT = 9.

    Entrada : señal -> shape (N,)
    Salida  : coeffs -> lista de 10 arrays
                coeffs[0] = cA9  (0–43 Hz)
                coeffs[1] = cD9  (43–86 Hz)
                ...
                coeffs[9] = cD1  (11025–22050 Hz)
    """
    return pywt.wavedec(señal, 'db4', level=NIVELES_DWT)
    # pywt.wavedec: aplica banco de filtros iterativo (paso-bajo H + paso-alto G) sobre la señal
    # 'db4': wavelet Daubechies-4; tiene 4 momentos nulos → suprime polinomios de grado ≤3 en cada detalle
    # level=9: realiza 9 iteraciones; en cada una la banda de aproximacion se divide a la mitad
    # retorna lista [cA9, cD9, cD8, ..., cD1] → 10 arrays de longitud decreciente (cada nivel tiene la mitad de muestras)


print('Modulo 2 OK — dwt_multirresolucion')  # confirma definicion correcta de la funcion

## Modulo 3 — Descriptores bioacusticos

Tres descriptores escalares por segmento, derivados de la DWT db4 (9 niveles).

| Descriptor | Banda | Fenomeno capturado |
|---|---|---|
| Jitter relativo | D7–D9 (43–344 Hz) | Variabilidad ciclo a ciclo de F0 |
| Shimmer relativo | D1–D4 (1378–22050 Hz) | Variabilidad de amplitud (ruido glotico) |
| Energía relativa | D1–D4 / total | Proporcion de energia en altas frecuencias |

In [ ]:
def _idx(j: int) -> int:
    """Convierte número de nivel Dj al índice en la lista de pywt.wavedec.
    Con NIVELES_DWT=9: D1→9, D4→6, D7→3, D9→1."""
    return NIVELES_DWT + 1 - j
    # pywt.wavedec invierte el orden: D1 (alta frec) queda al final de la lista
    # formula: D1 → 9+1-1=9  |  D9 → 9+1-9=1  |  D4 → 9+1-4=6


def reconstruir_banda(coeffs: list, niveles: list) -> np.ndarray:
    """
    Reconstruye la señal filtrando solo los niveles de detalle indicados.
    Anula todos los demás coeficientes y aplica pywt.waverec.

    niveles: lista de enteros, ej. [7, 8, 9] para jitter, [1, 2, 3, 4] para shimmer.
    """
    coeffs_filt = [np.zeros_like(c) for c in coeffs]
    # crea una lista de arrays de ceros con la misma forma que cada coeficiente original
    # todos los niveles empiezan en cero; solo se activaran los indicados en 'niveles'

    for j in niveles:
        idx = _idx(j)                          # traduce el numero de nivel Dj al indice en la lista
        coeffs_filt[idx] = coeffs[idx].copy()  # copia los coeficientes reales del nivel j; .copy() evita modificar coeffs original

    return pywt.waverec(coeffs_filt, 'db4')
    # pywt.waverec: DWT inversa que reconstruye la señal temporal solo con las bandas activadas
    # equivale a un filtro paso-banda en dominio wavelet (la ortogonalidad de db4 garantiza separacion exacta)


def calcular_jitter(coeffs: list, sr: int = SR) -> float:
    """
    Jitter relativo: variabilidad ciclo a ciclo de la frecuencia fundamental.
    Opera sobre la banda D7-D9 (43–344 Hz), que contiene la F0 pura
    tras filtrar armónicos y ruido por ortogonalidad de db4.

    Retorna: jitter relativo (adimensional, 0 si hay menos de 3 picos).
    """
    banda = reconstruir_banda(coeffs, [7, 8, 9])
    # reconstruye la señal filtrada a 43–344 Hz (zona de F0 de la voz, sin armonicos ni ruido glotico)

    dist_min = max(2, int(sr / 344))
    # distancia minima entre picos en muestras: sr/344 Hz ≈ 128 muestras @ 44100 Hz
    # equivale a la duracion de un ciclo a la frecuencia maxima de la banda (344 Hz)
    # max(2,...) evita dist_min=0 o 1 en casos extremos

    peaks, _ = find_peaks(banda, distance=dist_min, height=0)
    # find_peaks: detecta indices de maximos locales positivos separados al menos dist_min muestras
    # distance=dist_min: impide contar dos picos del mismo ciclo glotico como ciclos separados
    # height=0: solo considera picos por encima del cero (semi-ciclos positivos de la onda)
    # el segundo valor retornado (_) son las propiedades de los picos, que no necesitamos

    if len(peaks) < 3:
        return 0.0
    # se necesitan al menos 3 picos para calcular 2 periodos y luego 1 diferencia de periodos

    periodos = np.diff(peaks).astype(np.float64)
    # np.diff(peaks): distancias en muestras entre picos consecutivos → representan los periodos T_i
    # .astype(float64): convierte a flotante para las divisiones posteriores sin truncamiento

    return float(np.mean(np.abs(np.diff(periodos))) / (np.mean(periodos) + 1e-9))
    # np.diff(periodos): variaciones consecutivas de periodo → |T_{i+1} - T_i|
    # np.mean(np.abs(...)): promedia esas variaciones (numerador del jitter relativo)
    # / (np.mean(periodos) + 1e-9): divide por el periodo medio para hacerlo adimensional; epsilon evita /0
    # float(): convierte el numpy scalar a Python float nativo


def calcular_shimmer(coeffs: list, sr: int = SR) -> float:
    """
    Shimmer relativo: variabilidad de amplitud en la banda de alta frecuencia.
    Opera sobre la banda D1-D4 (1378–22050 Hz), que aísla el ruido glótico
    y el soplo por escape de aire.

    Retorna: shimmer relativo (adimensional, 0 si no hay suficientes ventanas).
    """
    banda = reconstruir_banda(coeffs, [1, 2, 3, 4])
    # reconstruye la señal filtrada a 1378–22050 Hz (zona de ruido glotico y soplosidad)

    envolvente = np.abs(hilbert(banda))
    # hilbert(banda): calcula la señal analitica z(t) = banda(t) + j·HT[banda(t)]
    # np.abs(...): magnitud de la señal analitica = envolvente instantanea A(t)
    # A(t) representa la amplitud del ruido glotico en cada instante

    ventana = max(1, int(sr * 0.01))
    # ventana de 10 ms en muestras: 44100 * 0.01 = 441 muestras
    # 10 ms ≈ duracion de un ciclo glotico medio (F0 tipica entre 100 y 200 Hz)
    # max(1,...) evita ventana de 0 muestras si sr fuera muy pequeno

    n_ventanas = len(envolvente) // ventana
    # cuantas ventanas completas caben en la envolvente (division entera, descarta el ultimo fragmento incompleto)

    if n_ventanas < 3:
        return 0.0
    # se necesitan al menos 3 ventanas para calcular 2 amplitudes y luego 1 diferencia

    amplitudes = np.array([
        np.max(envolvente[i * ventana : (i + 1) * ventana])  # pico de envolvente en la ventana i-esima de 10 ms
        for i in range(n_ventanas)                             # itera sobre todas las ventanas completas
    ])
    # amplitudes: vector con el maximo de A(t) en cada ventana → representa la amplitud pico de cada ciclo

    return float(np.mean(np.abs(np.diff(amplitudes))) / (np.mean(amplitudes) + 1e-9))
    # np.diff(amplitudes): variaciones de amplitud entre ventanas consecutivas |A_{i+1} - A_i|
    # np.mean(np.abs(...)): promedia esas variaciones (numerador del shimmer relativo)
    # / (np.mean(amplitudes) + 1e-9): normaliza por la amplitud media para hacerlo adimensional; epsilon evita /0


def calcular_energia_espectral(coeffs: list) -> float:
    """
    Energía relativa en la banda D1-D4 respecto a la energía total.
    E_rel = sum(D1² + D2² + D3² + D4²) / sum(todos los arrays²)

    Valores altos → mayor proporción de energía en altas frecuencias
    (indicativo de ruido glótico / soplosidad parkinsónica).
    """
    energia_total = sum(np.sum(c ** 2) for c in coeffs)
    # Teorema de Parseval para wavelets: la energia de la señal = suma de cuadrados de todos los coeficientes
    # c**2: eleva al cuadrado cada coeficiente (energia instantanea); np.sum suma todo el array
    # sum(...): acumula la energia de los 10 niveles (cA9 + cD1-cD9)

    if energia_total < 1e-12:
        return 0.0
    # umbral para considerar la señal numericamente silenciosa y evitar division por cero

    energia_alta = sum(np.sum(coeffs[_idx(j)] ** 2) for j in [1, 2, 3, 4])
    # suma la energia solo de los niveles D1, D2, D3, D4 (1378–22050 Hz)
    # _idx(j): traduce el numero de nivel al indice correcto en la lista de coeffs
    # en PD se espera mayor proporcion de energia en estas bandas por el ruido glotico adicional

    return float(energia_alta / energia_total)
    # cociente adimensional entre 0 y 1 que indica que fraccion de la energia total esta en altas frecuencias
    # float(): convierte el numpy scalar a Python float nativo para compatibilidad con pandas


print('Modulo 3 OK — calcular_jitter | calcular_shimmer | calcular_energia_espectral')

## Modulo 4 — Visualizaciones

Cuatro gráficos comparativos PD vs HC:

1. Espectrograma de **banda ancha** — alta resolución temporal (pulsos glóticos)
2. Espectrograma de **banda angosta** — alta resolución frecuencial (armónicos)
3. **Escalograma** DWT — mapa tiempo × nivel D1–D9
4. **Dispersión** jitter vs shimmer — un punto por sujeto

In [ ]:
plt.rcParams.update({
    'figure.dpi'      : 110,        # resolucion de figura en pantalla: 110 puntos por pulgada
    'figure.facecolor': '#0f1117',  # fondo exterior de la figura: azul muy oscuro (tema oscuro)
    'axes.facecolor'  : '#1a1d2e',  # fondo interior del area de grafico: azul oscuro
    'text.color'      : 'white',    # color de todo el texto generado por matplotlib
    'axes.labelcolor' : 'white',    # color de etiquetas de ejes (xlabel/ylabel)
    'xtick.color'     : 'white',    # color de marcas y numeros en el eje X
    'ytick.color'     : 'white',    # color de marcas y numeros en el eje Y
})
# plt.rcParams: diccionario global de matplotlib que controla el estilo visual de TODAS las figuras del notebook


def graficar_espectrograma_ancho(señal_pd, señal_hc, sr=SR):
    """
    Espectrograma de banda ancha: ventana corta (~5 ms), alta resolución temporal.
    Permite ver pulsos glóticos individuales y fluctuaciones de amplitud (shimmer).
    """
    n_fft = 256           # tamaño de la FFT = 256 muestras ≈ 5.8 ms @ 44100 Hz → ventana corta → alta resolucion temporal
    hop   = n_fft // 4    # salto entre ventanas = 64 muestras (75% de solapamiento entre ventanas sucesivas)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    # crea una figura con 2 subplots en fila: izquierda para PD, derecha para HC
    # figsize=(14,4): 14 pulgadas de ancho, 4 de alto

    for ax, señal, titulo in zip(axes, [señal_pd, señal_hc], ['PD', 'HC']):
        # itera simultaneamente sobre: los ejes, las señales y los titulos

        S    = librosa.stft(señal.astype(np.float32), n_fft=n_fft, hop_length=hop)
        # STFT: Short-Time Fourier Transform; divide la señal en ventanas solapadas y aplica FFT a cada una
        # resultado: matriz compleja shape (n_fft/2+1, n_frames)
        # .astype(float32): reduce uso de memoria sin perder precision perceptible

        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
        # np.abs(S): magnitud del espectro complejo (modulo de cada bin frecuencial)
        # amplitude_to_db: convierte amplitud lineal a decibelios: 20·log10(|S|/ref)
        # ref=np.max: normaliza respecto al maximo → rango tipico de -80 a 0 dB

        librosa.display.specshow(S_db, sr=sr, hop_length=hop,
                                 x_axis='time', y_axis='hz', ax=ax, cmap='magma')
        # dibuja S_db como imagen de calor con ejes calibrados en tiempo [s] y frecuencia [Hz]
        # cmap='magma': paleta oscuro→amarillo; buena percepcion en tema oscuro y en blanco y negro

        ax.set_title(f'Espectrograma banda ancha — {titulo}')  # titulo del subplot con el grupo (PD o HC)
        ax.set_ylim(0, 5000)  # limita el eje Y a 0-5000 Hz (cubre F0 y los primeros armonicos de voz)

    plt.suptitle('Banda ancha: alta resolución temporal (pulsos glóticos)', color='white')
    # titulo general que encabeza la figura completa (ambos subplots)
    plt.tight_layout()  # ajusta margenes automaticamente para que los subplots no se superpongan ni recorten
    plt.savefig('espectrograma_banda_ancha.png', bbox_inches='tight')
    # guarda la figura como PNG; bbox_inches='tight' recorta el espacio en blanco exterior
    plt.show()  # renderiza y muestra la figura en la celda del notebook


def graficar_espectrograma_angosto(señal_pd, señal_hc, sr=SR):
    """
    Espectrograma de banda angosta: ventana larga (~40 ms), alta resolución frecuencial.
    Permite separar armónicos individuales y visualizar ondulaciones de F0 (jitter).
    """
    n_fft = 2048          # tamaño FFT = 2048 muestras ≈ 46 ms @ 44100 Hz → ventana larga → alta resolucion frecuencial
    hop   = n_fft // 8    # salto = 256 muestras (87.5% de solapamiento); mas solapamiento → curvas de F0 mas suaves

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    # mismo layout que el espectrograma de banda ancha: 2 subplots lado a lado

    for ax, señal, titulo in zip(axes, [señal_pd, señal_hc], ['PD', 'HC']):

        S    = librosa.stft(señal.astype(np.float32), n_fft=n_fft, hop_length=hop)
        # STFT con ventana larga: cada bin frecuencial cubre ~21 Hz (mayor resolucion frecuencial)
        # pero menor resolucion temporal: cada columna representa ~46 ms de audio

        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
        # convierte la magnitud del espectro a escala logaritmica (dB) para mejorar contraste visual

        librosa.display.specshow(S_db, sr=sr, hop_length=hop,
                                 x_axis='time', y_axis='hz', ax=ax, cmap='magma')
        # con ventana larga los armonicos aparecen como lineas horizontales finas y bien separadas

        ax.set_title(f'Espectrograma banda angosta — {titulo}')
        ax.set_ylim(0, 2000)  # limita a 0-2000 Hz para ver los primeros armonicos con mayor detalle

    plt.suptitle('Banda angosta: alta resolución frecuencial (armónicos)', color='white')
    plt.tight_layout()
    plt.savefig('espectrograma_banda_angosta.png', bbox_inches='tight')
    plt.show()


def graficar_escalograma(coeffs_pd, coeffs_hc):
    """
    Escalograma DWT: mapa 2D tiempo × nivel D1–D9, magnitud como intensidad cromática.
    Muestra la distribución de energía multirresolución de la db4.
    Se espera: bandas estables en HC, dispersión en zonas agudas en PD.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # figura con 2 subplots horizontales, un poco mas altos
    etiquetas_y = [f'D{j}' for j in range(1, NIVELES_DWT + 1)]
    # genera la lista ['D1','D2',...,'D9'] para etiquetar el eje Y del escalograma

    for ax, coeffs, titulo in zip(axes, [coeffs_pd, coeffs_hc], ['PD', 'HC']):

        longitud_ref = len(coeffs[_idx(5)])
        # usa la longitud del nivel D5 como referencia de interpolacion
        # D5 esta en el centro de los 9 niveles → longitud moderada, equilibra resolucion vs memoria

        matriz = []  # acumulara un array por nivel de detalle (9 filas en total)
        for j in range(1, NIVELES_DWT + 1):

            d = np.abs(coeffs[_idx(j)])
            # magnitud de los coeficientes del nivel Dj → energia local en esa banda frecuencial

            d_interp = np.interp(
                np.linspace(0, 1, longitud_ref),  # posiciones destino: longitud_ref puntos uniformes en [0,1]
                np.linspace(0, 1, len(d)),         # posiciones origen: len(d) puntos uniformes en [0,1]
                d                                   # valores a interpolar
            )
            # np.interp: interpolacion lineal que lleva todos los niveles a la misma longitud
            # necesario porque cada nivel Dj tiene distinta cantidad de coeficientes (se divide a la mitad por nivel)

            matriz.append(d_interp)  # agrega la fila interpolada a la lista de filas

        matriz = np.array(matriz)
        # convierte la lista a matriz 2D de shape (9, longitud_ref): filas=niveles, columnas=tiempo

        im = ax.imshow(matriz, aspect='auto', origin='lower',
                       cmap='magma', interpolation='nearest')
        # imshow: muestra la matriz como imagen de calor
        # aspect='auto': ajusta la proporcion de aspecto automaticamente al tamaño del subplot
        # origin='lower': D1 queda abajo (alta frec) y D9 arriba (baja frec), consistente con ejes frecuenciales
        # interpolation='nearest': sin suavizado; muestra los coeficientes exactos sin difuminar

        ax.set_yticks(range(NIVELES_DWT))          # posicion de las marcas en Y: 0 a 8
        ax.set_yticklabels(etiquetas_y)             # etiqueta cada marca con 'D1' a 'D9'
        ax.set_xlabel('Tiempo (coeficientes interpolados)')  # eje X en unidades de coeficientes (no en segundos reales)
        ax.set_ylabel('Nivel DWT')                  # eje Y = nivel de descomposicion wavelet
        ax.set_title(f'Escalograma db4 — {titulo}') # titulo del subplot con el grupo (PD o HC)
        plt.colorbar(im, ax=ax, label='|coef.|')
        # barra de color lateral: mapea intensidad cromatica → magnitud absoluta del coeficiente wavelet

    plt.suptitle('Escalograma wavelet: densidad de energía multirresolución', color='white')
    plt.tight_layout()
    plt.savefig('escalograma_wavelet.png', bbox_inches='tight')
    plt.show()


def graficar_dispersion_jitter_shimmer(df: pd.DataFrame):
    """
    Mapa de dispersión jitter vs shimmer, un punto por sujeto, color por grupo.
    Entrada: DataFrame con columnas 'jitter_mean', 'shimmer_mean', 'grupo'
             (grupo = 1 para PD, 0 para HC).
    """
    fig, ax = plt.subplots(figsize=(8, 6))  # figura unica de 8x6 pulgadas para el scatter

    colores   = {0: '#4fc3f7', 1: '#ef5350'}
    # diccionario de colores por grupo: HC=azul claro (#4fc3f7), PD=rojo (#ef5350)

    etiquetas = {0: 'HC (sano)', 1: 'PD (Parkinson)'}
    # texto que aparece en la leyenda para cada grupo

    for grupo, sub in df.groupby('grupo'):
        # df.groupby('grupo'): separa el DataFrame en subgrupos segun el valor de 'grupo' e itera sobre cada uno
        # grupo: valor del grupo (0 o 1); sub: sub-DataFrame con solo las filas de ese grupo

        ax.scatter(sub['jitter_mean'], sub['shimmer_mean'],
                   c=colores[grupo],        # color del punto segun el grupo
                   label=etiquetas[grupo],  # texto de la leyenda para este grupo
                   alpha=0.75,              # transparencia del 25%: permite ver puntos superpuestos
                   s=60,                    # tamaño del marcador en puntos cuadrados
                   edgecolors='white',      # borde blanco alrededor de cada punto (mejor contraste en tema oscuro)
                   linewidths=0.4)          # grosor del borde blanco

    ax.set_xlabel('Jitter relativo (D7–D9)')   # eje X: jitter promedio por sujeto (variabilidad de F0)
    ax.set_ylabel('Shimmer relativo (D1–D4)')  # eje Y: shimmer promedio por sujeto (variabilidad de amplitud)
    ax.set_title('Distribución jitter-shimmer: PD vs HC')
    ax.legend()         # muestra la leyenda con los colores de cada grupo
    plt.tight_layout()
    plt.savefig('dispersion_jitter_shimmer.png', bbox_inches='tight')
    plt.show()


print('Modulo 4 OK — graficar_espectrograma_ancho | graficar_espectrograma_angosto | graficar_escalograma | graficar_dispersion_jitter_shimmer')

## Modulo 5 — Construccion del dataset

Procesa la vocal `/a/` de PC-GITA y exporta un CSV de 100 filas para Orange.

```
PC-GITA_per_task_44100Hz/modulated vowels/
├── pd/
│   └── A/   ← 50 archivos PD  (etiqueta 1)
└── hc/
    └── A/   ← 50 archivos HC  (etiqueta 0)
```

In [ ]:
DATA_DIR  = Path(r'C:\Users\mateo\OneDrive\Desktop\UNI Sheiße\S&S\PC-GITA_per_task_44100Hz\PC-GITA_per_task_44100Hz\modulated vowels')
# Path(): convierte el string en objeto de ruta multiplataforma
# r'...': raw string → la barra \ no es interpretada como secuencia de escape

LABEL_MAP = {'pd': 1, 'hc': 0}
# mapea nombre de subcarpeta → etiqueta numerica: PD=1 (enfermo), HC=0 (control sano)
# convencion binaria estandar para clasificadores supervisados

VOCAL     = 'A'
# subcarpeta dentro de pd/ y hc/ con los .wav de la vocal /a/ sostenida
# PC-GITA organiza por tarea; A = vocal /a/ (la mas usada en estudios de Parkinson)

print(f'DATA_DIR : {DATA_DIR}')
# imprime la ruta base para verificar visualmente que es correcta antes de procesar

print(f'PD existe: {(DATA_DIR / "pd" / VOCAL).exists()}')
# (DATA_DIR / "pd" / VOCAL): operador / de Path concatena subcarpetas sin importar el SO
# .exists(): True si la carpeta existe en el sistema de archivos en este momento

print(f'HC existe: {(DATA_DIR / "hc" / VOCAL).exists()}')
# verifica que la carpeta de controles sanos tambien existe antes de intentar leerla

In [ ]:
def construir_dataset(data_dir: Path = DATA_DIR,
                      vocal: str = VOCAL) -> pd.DataFrame:
    """
    Itera sobre pd/A/ y hc/A/, procesa cada .wav y agrega los descriptores
    bioacústicos a nivel de sujeto (promedio y std sobre todos los segmentos).

    Salida: DataFrame con 100 filas y columnas:
        sujeto_id, grupo, jitter_mean, jitter_std,
        shimmer_mean, shimmer_std, energia_mean, energia_std
    """
    registros = []
    # lista vacia que acumulara un diccionario de features por cada sujeto procesado

    for clase, etiqueta in LABEL_MAP.items():
        # itera sobre {'pd': 1, 'hc': 0}; clase='pd'/'hc', etiqueta=1/0

        carpeta = data_dir / clase / vocal
        # construye la ruta completa a la carpeta de esa clase: .../modulated vowels/pd/A/

        if not carpeta.exists():
            print(f"[AVISO] Carpeta no encontrada: {carpeta}  — omitiendo.")
            continue
        # si la carpeta no existe, avisa y salta a la siguiente clase sin detener la ejecucion

        archivos = sorted(carpeta.glob("*.wav"))
        # glob("*.wav"): busca todos los archivos con extension .wav en la carpeta
        # sorted(): ordena por nombre para que el orden sea reproducible en cada ejecucion

        print(f"  {clase}/{vocal}: {len(archivos)} archivos")
        # informa cuantos sujetos se encontraron en esta clase

        for i, archivo in enumerate(archivos):
            # itera sobre cada archivo .wav; un archivo = un sujeto de la base de datos

            sujeto_id = archivo.stem
            # .stem: nombre del archivo sin la extension (ej. 'AVPEPUDEA0001_a')
            # es el identificador unico del sujeto que Orange usara para LOSO

            try:
                señal, sr = cargar_señal(archivo)
                # carga el .wav como array numpy float64 normalizado en [-1,1]

                señal     = recortar_silencios(señal, sr)
                # elimina tramos de silencio para que los segmentos contengan solo voz activa

                segmentos = segmentar(señal, sr)
                # divide la señal en ventanas de 2 s con 50% de solapamiento

                jitters, shimmers, energias = [], [], []
                # listas para acumular los descriptores de cada segmento del mismo sujeto

                for seg in segmentos:
                    # itera sobre cada ventana de 2 s

                    coeffs = dwt_multirresolucion(seg)
                    # descompone el segmento en 10 niveles DWT db4 → [cA9, cD9, ..., cD1]

                    jitters.append(calcular_jitter(coeffs, sr))
                    # agrega el jitter relativo del segmento a la lista del sujeto

                    shimmers.append(calcular_shimmer(coeffs, sr))
                    # agrega el shimmer relativo del segmento

                    energias.append(calcular_energia_espectral(coeffs))
                    # agrega la fraccion de energia en altas frecuencias del segmento

                registros.append({
                    'sujeto_id'    : sujeto_id,          # identificador unico (nombre del archivo .wav sin extension)
                    'grupo'        : etiqueta,            # 1=PD, 0=HC; variable de clase para el clasificador
                    'jitter_mean'  : np.mean(jitters),   # jitter promedio sobre todos los segmentos del sujeto
                    'jitter_std'   : np.std(jitters),    # desvio estandar del jitter (variabilidad intra-sujeto)
                    'shimmer_mean' : np.mean(shimmers),  # shimmer promedio
                    'shimmer_std'  : np.std(shimmers),   # desvio estandar del shimmer
                    'energia_mean' : np.mean(energias),  # energia relativa promedio en altas frecuencias
                    'energia_std'  : np.std(energias),   # desvio estandar de la energia relativa
                })

            except Exception as e:
                print(f"  [ERROR] {archivo.name}: {e}")
                # captura cualquier error (archivo corrupto, señal demasiado corta, etc.) sin detener el loop

    if not registros:
        raise RuntimeError(
            "No se encontraron datos.\n"
            f"  Verificar DATA_DIR / LABEL_MAP / VOCAL.\n"
            f"  Ruta esperada: {data_dir / 'pd' / vocal}  y  {data_dir / 'hc' / vocal}"
        )
    # si la lista quedo vacia (ninguna carpeta valida o ningun .wav encontrado), lanza error descriptivo

    df = pd.DataFrame(registros)
    # convierte la lista de diccionarios en un DataFrame de pandas (una fila por sujeto)

    print(f"\nDataset construido: {len(df)} sujetos — "
          f"{(df['grupo'] == 1).sum()} PD / {(df['grupo'] == 0).sum()} HC")
    # informa el total de sujetos y el balance de clases (idealmente 50 PD y 50 HC)

    return df


def exportar_csv(df: pd.DataFrame, ruta: str = "features_pcgita.csv") -> None:
    """
    Exporta el DataFrame de sujetos a CSV para importar en Orange.
    Orange usará 'grupo' como variable de clase y 'sujeto_id' para LOSO.
    """
    df.to_csv(ruta, index=False)
    # .to_csv(): escribe el DataFrame en disco como CSV separado por comas
    # index=False: omite el indice numerico de pandas (Orange no lo necesita como columna)

    print(f"CSV exportado: {ruta}  ({len(df)} filas × {len(df.columns)} columnas)")
    # confirma la exportacion mostrando el nombre del archivo y las dimensiones del dataset


# Para correr:
# df = construir_dataset()
# exportar_csv(df)
print('Modulo 5 OK — construir_dataset | exportar_csv')

## Modulo 6 — Ejecución: construcción del dataset y graficación

In [ ]:
# ── 1. Dataset completo ──────────────────────────────────────────────────────
df = construir_dataset()
# llama a construir_dataset() con los valores por defecto (DATA_DIR y VOCAL='A')
# procesa todos los .wav de pd/A/ y hc/A/ y devuelve un DataFrame de 100 sujetos

exportar_csv(df)
# escribe df en 'features_pcgita.csv' en el directorio de trabajo actual del notebook

# ── 2. Señales de ejemplo para los espectrogramas y el escalograma ───────────
archivos_pd = sorted((DATA_DIR / 'pd' / VOCAL).glob('*.wav'))
# lista ordenada de todos los .wav del grupo PD; sorted() garantiza orden reproducible entre ejecuciones

archivos_hc = sorted((DATA_DIR / 'hc' / VOCAL).glob('*.wav'))
# lista ordenada de todos los .wav del grupo HC (controles sanos)

señal_pd, _ = cargar_señal(archivos_pd[0])
# carga el primer archivo PD como array numpy normalizado; _ descarta sr (ya conocemos SR=44100)

señal_pd     = recortar_silencios(señal_pd, SR)
# elimina tramos de silencio inicial/final de la señal PD antes de segmentar

señal_hc, _ = cargar_señal(archivos_hc[0])
# carga el primer archivo HC (control sano)

señal_hc     = recortar_silencios(señal_hc, SR)
# elimina silencios de la señal HC

# Tomamos el primer segmento completo de cada señal para la DWT
seg_pd = segmentar(señal_pd, SR)[0]
# segmentar() devuelve lista de ventanas de 2 s; [0] toma la primera (88200 muestras)

seg_hc = segmentar(señal_hc, SR)[0]
# idem para la señal HC

coeffs_pd = dwt_multirresolucion(seg_pd)
# calcula la DWT de 9 niveles del segmento PD → lista de 10 arrays necesaria para el escalograma

coeffs_hc = dwt_multirresolucion(seg_hc)
# idem para el segmento HC

print(f'Sujeto PD de ejemplo : {archivos_pd[0].stem}')
# muestra el ID del sujeto PD elegido como ejemplo (nombre del archivo sin extension)

print(f'Sujeto HC de ejemplo : {archivos_hc[0].stem}')
# muestra el ID del sujeto HC elegido como ejemplo

# ── 3. Gráficos ───────────────────────────────────────────────────────────────
graficar_espectrograma_ancho(seg_pd, seg_hc)
# compara PD vs HC con ventana corta (~5 ms): muestra pulsos gloticos y variabilidad temporal (shimmer)

graficar_espectrograma_angosto(seg_pd, seg_hc)
# compara PD vs HC con ventana larga (~46 ms): muestra armonicos separados y variaciones de F0 (jitter)

graficar_escalograma(coeffs_pd, coeffs_hc)
# mapa 2D tiempo x nivel DWT: visualiza donde se concentra la energia en cada banda frecuencial

graficar_dispersion_jitter_shimmer(df)
# scatter jitter vs shimmer con un punto por sujeto; usa el dataset completo de 100 sujetos